# Create an elastic network for 7tx0 (mac1, p43 space group)

Goal: create a real enm to share with developers

Code mostly lifted from [`proto-hessian.ipynb`](proto-hessian.ipynb)

Steps:

1. Use Mac1 P43 + ADPr [7TX0](https://www.rcsb.org/structure/7TX0), with two copies in the ASU. Group atoms by chain ID.
2. Pack the unit cell
3. Construct the contacts list for the ASU. Keep only contacts between groups (or with other ASUs).
4. Expand to P1 and add node positions
5. Save as csv table

### 1. Load pdb file and assign rigid groups

In [1]:
# !cd test_data && curl -O https://files.rcsb.org/download/7TX0.cif

import gemmi

coordinate_file = 'test_data/7TX0.cif'

st = gemmi.read_structure(coordinate_file)
st.setup_entities()  # supposed to be good practice

print('Unit Cell:',st.cell.parameters)
print('Space Group:',st.spacegroup_hm)

Unit Cell: (88.451, 88.451, 39.823, 90.0, 90.0, 90.0)
Space Group: P 43


In [2]:
# assign groups, save index of tls group for each atom in atom.tls_group_id
groups = [
    gemmi.Selection('//A;polymer'),
    gemmi.Selection('//B;polymer'),
]

tls_groups = []

# calculate the group center of mass, store as TLS group origin
for group_id, selection in enumerate(groups):
    model = selection.copy_model_selection(st[0])
    com = model.calculate_center_of_mass()
    g = gemmi.TlsGroup()
    g.id = f"TLS{group_id}"  # why is this a string? why can't I access num_id property?
    g.origin = com
    tls_groups.append(g)

st.meta.refinement[0].tls_groups = tls_groups

for group_id, selection in enumerate(groups):
    for model in selection.models(st):
        for chain in selection.chains(model):
            for residue in selection.residues(chain):
                for atom in selection.atoms(residue):
                    atom.tls_group_id = group_id

# question... does the tls_group_id index into the list of TLS groups? or does there need to be some other mapping?


### 3. Pack the unit cell

This modifies st.cell.images with extra pbc shifts if needed

In [3]:
# before packing
for j, im in enumerate(st.cell.images):
    print('sym_idx:', j + 1)
    print(im.mat)
    print(im.vec,'\n')

sym_idx: 1
<gemmi.Mat33 [0, -1, 0]
             [1, 0, 0]
             [0, 0, 1]>
<gemmi.Vec3(0, 0, 0.75)> 

sym_idx: 2
<gemmi.Mat33 [-1, 0, 0]
             [0, -1, 0]
             [0, 0, 1]>
<gemmi.Vec3(0, 0, 0.5)> 

sym_idx: 3
<gemmi.Mat33 [0, 1, 0]
             [-1, 0, 0]
             [0, 0, 1]>
<gemmi.Vec3(0, 0, 0.25)> 



In [4]:
# after packing (pbc shifts are different)
from goodvibes import enm

enm._pack_unit_cell(st, inplace=True)

for j, im in enumerate(st.cell.images):
    print('sym_idx:', j + 1)
    print(im.mat)
    print(im.vec,'\n')

sym_idx: 1
<gemmi.Mat33 [0, -1, 0]
             [1, 0, 0]
             [0, 0, 1]>
<gemmi.Vec3(1, 0, 0.75)> 

sym_idx: 2
<gemmi.Mat33 [-1, 0, 0]
             [0, -1, 0]
             [0, 0, 1]>
<gemmi.Vec3(1, 1, 0.5)> 

sym_idx: 3
<gemmi.Mat33 [0, 1, 0]
             [-1, 0, 0]
             [0, 0, 1]>
<gemmi.Vec3(0, 1, 0.25)> 



### 2. Construct the contacts list

In [5]:
df = enm._find_contacts(
   st,
   distance_cutoff=4.0,
   include_h=False,
)

def address_from_cra_string(cra):
    chain, residue, atom = cra.split('/')
    resname, seqid = residue.split()
    if '.' in atom:
        atom, altloc = atom.split('.')
    else:
        altloc = '\x00'
    addr =  gemmi.AtomAddress(chain, gemmi.SeqId(seqid), resname, atom, altloc)
    return addr

df['group_id1'] = -1
df['group_id2'] = -1

# add group ID columns to dataframe
for row in df.itertuples():
    cra1 = st[0].find_cra(address_from_cra_string(row.cra1))
    cra2 = st[0].find_cra(address_from_cra_string(row.cra2))
    df.at[row.Index, 'group_id1'] = cra1.atom.tls_group_id
    df.at[row.Index, 'group_id2'] = cra2.atom.tls_group_id

df['external'] = False

# add a column that indicates whether the contact is external (between different groups)
for row in df.itertuples():
    if row.group_id1 == -1 or row.group_id2 == -1:
        df.at[row.Index, 'external'] = False
    elif (row.sym_idx1 == row.sym_idx2) and (row.pbc_shift1 == row.pbc_shift2):
        # same ASU, check if the two residues have the same flag (i.e. belong to the same rigid group)
        df.at[row.Index, 'external'] = (row.group_id1 != row.group_id2)
    else:
        # different ASU, any group flag (not null)
        df.at[row.Index, 'external'] = True

# create a new dataframe that contains only the external contacts, dropping the 'external' column, reset index.
external_df = df[df['external']].drop(columns=['external']).reset_index(drop=True)
external_df

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,group_id1,group_id2
0,A/GLY 8/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
1,A/GLY 8/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
2,A/GLY 8/O,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
3,A/TYR 9/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
4,A/TYR 9/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0
...,...,...,...,...,...,...,...,...
612,B/LEU 169/CD1,B/ASP 22/OD1,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1
613,B/LEU 169/CD1,B/ASP 22/OD2,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1
614,B/LEU 169/CD2,B/ILE 23/CG2,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1
615,B/LEU 169/CD2,B/ALA 52/CB,0,1,"(0, 0, 0)","(-1, 0, -1)",1,1


In [6]:
# group by columns sym_idx1, sym_idx2, pbc_shift1, pbc_shift2, group_id1, group_id2 and count the number of contacts
grouped_df = external_df.groupby(['sym_idx1', 'sym_idx2', 'pbc_shift1', 'pbc_shift2', 'group_id1', 'group_id2']).size()
grouped_df

sym_idx1  sym_idx2  pbc_shift1  pbc_shift2   group_id1  group_id2
0         0         (0, 0, 0)   (0, 0, -1)   0          0             9
                                             1          1            21
                                (0, 0, 0)    0          1            83
                                (0, 0, 1)    0          0             9
                                             1          1            21
          1         (0, 0, 0)   (-1, 0, -1)  1          1            70
                                (0, 0, -1)   0          0            85
          2         (0, 0, 0)   (0, -1, -1)  0          1            18
                                             1          0            64
                                (0, -1, 0)   0          1            64
                                             1          0            18
          3         (0, 0, 0)   (0, -1, 0)   1          1            70
                                (0, 0, 0)    0          0            8

### 3. Expand to P1

In [7]:
import pandas as pd

external_df_expanded = enm._symmetry_expand(external_df, st.cell.images)

def transform_position(st, sym_idx, pbc_shift, pos):
    if sym_idx == 0:
        image_transform = gemmi.Transform()
    else:
        image_transform = st.cell.images[sym_idx - 1]
    pbc_transform = gemmi.Transform(gemmi.Mat33(), gemmi.Vec3(*pbc_shift))
    t = pbc_transform @ image_transform
    return st.cell.orthogonalize(gemmi.Fractional(t.apply(st.cell.fractionalize(pos))))

external_df_expanded['r1'] = pd.Series([None] * len(external_df_expanded), dtype=object)
external_df_expanded['r2'] = pd.Series([None] * len(external_df_expanded), dtype=object)

for row in external_df_expanded.itertuples():
    cra1 = st[0].find_cra(address_from_cra_string(row.cra1))
    cra2 = st[0].find_cra(address_from_cra_string(row.cra2))
    pos1 = transform_position(st, row.sym_idx1, row.pbc_shift1, cra1.atom.pos)
    pos2 = transform_position(st, row.sym_idx2, row.pbc_shift2, cra2.atom.pos)
    external_df_expanded.at[row.Index, 'r1'] = (pos1.x, pos1.y, pos1.z)
    external_df_expanded.at[row.Index, 'r2'] = (pos2.x, pos2.y, pos2.z)

external_df_expanded

,cra1,cra2,sym_idx1,sym_idx2,pbc_shift1,pbc_shift2,group_id1,group_id2,r1,r2
0,A/GLY 8/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0,"(36.19777, 35.47873, 15.606150000000001)","(33.5479, 38.297419999999995, 15.76611)"
1,A/GLY 8/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0,"(36.59681, 36.19113, 14.68052)","(34.09091, 38.94922, 14.69258)"
2,A/GLY 8/O,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0,"(36.59681, 36.19113, 14.68052)","(33.5479, 38.297419999999995, 15.76611)"
3,A/TYR 9/C,A/PHE 156/CZ,0,3,"(0, 0, 0)","(0, 0, 0)",0,0,"(33.53765, 35.06444, 13.83823)","(33.5479, 38.297419999999995, 15.76611)"
4,A/TYR 9/O,A/PHE 156/CE2,0,3,"(0, 0, 0)","(0, 0, 0)",0,0,"(32.68961, 35.29065, 14.70957)","(34.09091, 38.94922, 14.69258)"
...,...,...,...,...,...,...,...,...,...,...
2208,B/LEU 169/CD1,B/ASP 22/OD1,3,0,"(0, 0, 0)","(0, 1, 0)",1,1,"(15.612220000000002, 79.41917, -4.556659999999...","(17.4578, 79.14677999999999, -7.70419)"
2209,B/LEU 169/CD1,B/ASP 22/OD2,3,0,"(0, 0, 0)","(0, 1, 0)",1,1,"(15.612220000000002, 79.41917, -4.556659999999...","(15.84539, 77.71637999999999, -7.72701)"
2210,B/LEU 169/CD2,B/ILE 23/CG2,3,0,"(0, 0, 0)","(0, 1, 0)",1,1,"(17.03124, 81.20943, -3.626659999999998)","(19.34131, 83.05068999999999, -5.65706)"
2211,B/LEU 169/CD2,B/ALA 52/CB,3,0,"(0, 0, 0)","(0, 1, 0)",1,1,"(17.03124, 81.20943, -3.626659999999998)","(18.87069, 78.58950999999999, -1.90476)"


### 5. Save the enm as a csv file

In [ ]:
external_df_expanded.to_csv('results/7tx0-enm-expanded.csv', index=False)
external_df.to_csv('results/7tx0-enm-asu.csv', index=False)